# AgentArk RL Tutorial

This Colab has two parts.

## Part 1: AgentArk RL Server Tutorial

This part is independent of any RL framework. It runs on Colab CPU and demonstrates the server-side contract used by RL trainers and debugging clients:

1. Clone the public AgentArk repository from GitHub and install its Python 3.10 environment.
2. Download the Linux AgentArk Unity runtime.
3. Start the framework-independent env server.
4. Warm up a small pool of Unity envs.
5. Run a framework-free parallel client over `acquire_start` / `step` / `release`, pin at least two tasks, and visualize screenshots returned by multiple env observations.

## Part 2: RL Framework Integration Notes

This part points to the maintained ms-swift and VERL runbooks for training outside Colab. Full multimodal agentic RL training needs a separate trainer environment, a suitable multimodal model checkpoint, and enough resources for rollout plus training, so the notebook does not run that job directly.

The key architectural point is that Part 1 is reusable: the AgentArk server side is RL-framework independent, and future trainers can use the same HTTP server API.


In [ ]:
# @title Step 1: Clone AgentArk and download the Linux runtime { display-mode: "form" }
# @markdown Clones the public AgentArk repository, creates a Python 3.10 virtual environment, downloads the Unity runtime, and installs AgentArk in editable mode.

AGENTARK_REPO_URL = "https://github.com/P90-RushB/AgentArk.git"  # @param {type:"string"}
AGENTARK_BRANCH = "main"  # @param {type:"string"}
ENV_ZIP_URL = "https://huggingface.co/datasets/P90-RushB/AgentArk/resolve/main/artifacts/envs/1.0.1/linux/AgentArk-env-1.0.1-linux.zip?download=true"  # @param {type:"string"}
FORCE_REINSTALL = False  # @param {type:"boolean"}

import shutil
import subprocess
import sys
from pathlib import Path

%cd /content

CONTENT_ROOT = Path("/content")
AGENTARK_ROOT = CONTENT_ROOT / "AgentArk"
VENV_DIR = CONTENT_ROOT / "agentark_env"
VENV_PY = VENV_DIR / "bin" / "python"
ENV_ZIP = CONTENT_ROOT / "AgentArk-env-1.0.1-linux.zip"
ENV_ROOT = CONTENT_ROOT / "AgentArk-env-1.0.1-linux"


def run(cmd):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)


print("Installing OS packages...")
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "git", "unzip", "wget", "xvfb", "python3.10", "python3.10-venv"])

if FORCE_REINSTALL:
    print("Removing previous /content installation...")
    shutil.rmtree(AGENTARK_ROOT, ignore_errors=True)
    shutil.rmtree(VENV_DIR, ignore_errors=True)
    shutil.rmtree(ENV_ROOT, ignore_errors=True)
    if ENV_ZIP.exists():
        ENV_ZIP.unlink()

if not VENV_DIR.exists():
    run(["python3.10", "-m", "venv", VENV_DIR])
else:
    print("Using existing virtual environment:", VENV_DIR)

if not AGENTARK_ROOT.exists():
    branch = AGENTARK_BRANCH.strip()
    clone_cmd = ["git", "clone", "--depth", "1"]
    if branch:
        clone_cmd.extend(["--branch", branch])
    clone_cmd.extend([AGENTARK_REPO_URL, AGENTARK_ROOT])
    run(clone_cmd)
else:
    print("Using existing AgentArk checkout:", AGENTARK_ROOT)

if not (AGENTARK_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(f"Expected {AGENTARK_ROOT / 'pyproject.toml'} after cloning AgentArk")

if not ENV_ROOT.exists():
    run(["wget", "-q", "-O", ENV_ZIP, ENV_ZIP_URL])
    run(["unzip", "-q", "-o", ENV_ZIP, "-d", CONTENT_ROOT])
else:
    print("Using existing Unity runtime:", ENV_ROOT)

exe = ENV_ROOT / "AgentArk.x86_64"
if not exe.exists():
    raise FileNotFoundError(f"Expected Unity executable at {exe}")
exe.chmod(0o755)

print("Installing AgentArk Python package...")
run([VENV_PY, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([VENV_PY, "-m", "pip", "install", "-q", "-e", AGENTARK_ROOT])

print("Installing notebook helper packages...")
run([sys.executable, "-m", "pip", "install", "-q", "omegaconf"])

print("AgentArk root:", AGENTARK_ROOT)
print("Unity runtime:", ENV_ROOT)
print("Python:", VENV_PY)


## Configure the runtime

The env server process and the framework-free client run in the AgentArk Python 3.10 environment. The config below uses a tiny pool suitable for Colab. Real RL training usually increases `warmup.num_envs` to match the expected rollout concurrency.

In [ ]:
# @title Step 2: Write `.env` and create a small Colab runtime config { display-mode: "form" }

NUM_ENVS = 2  # @param {type:"integer"}
POOL_SIZE = 2  # @param {type:"integer"}
GALLIUM_NUM_THREADS = 4  # @param {type:"integer"}

import os
from pathlib import Path
from omegaconf import OmegaConf

AGENTARK_ROOT = Path("/content/AgentArk")
ENV_ROOT = Path("/content/AgentArk-env-1.0.1-linux")
MOD_PATH = ENV_ROOT / "AgentArk_Data" / "Resources" / "Mods"
TASK_STORE_PATH = MOD_PATH / "all_tasks"
RUNTIME_POOL_ROOT = Path("/tmp/agentark_runtime_pool")

env_values = {
    "AGENTARK_ENV_PATH": str(ENV_ROOT / "AgentArk.x86_64"),
    "AGENTARK_MOD_PATH": str(MOD_PATH),
    "AGENTARK_TASK_STORE_PATH": str(TASK_STORE_PATH),
    "AGENTARK_RUNTIME_TEMPLATE_ROOT": str(ENV_ROOT),
    "AGENTARK_RUNTIME_POOL_ROOT": str(RUNTIME_POOL_ROOT),
    "MLAGENTS_PYTHON_BIN": "/content/agentark_env/bin/python",
}
os.environ.update(env_values)

env_text = "\n".join(f"{k}={v}" for k, v in env_values.items()) + "\n"
(AGENTARK_ROOT / ".env").write_text(env_text, encoding="utf-8")

example_cfg = AGENTARK_ROOT / "config" / "ark_env" / "agentark_runtime_config.example.yaml"
colab_cfg = AGENTARK_ROOT / "config" / "ark_env" / "agentark_runtime_config.colab.yaml"
cfg = OmegaConf.load(example_cfg)
cfg.warmup.num_envs = int(NUM_ENVS)
cfg.warmup.close_envs_after_warmup = False
cfg.warmup.step_once = False
cfg.env_cfg.runtime_sandbox.pool_size = int(max(POOL_SIZE, NUM_ENVS))
cfg.env_cfg.env_config_overrides.virtual_display = True
cfg.env_cfg.env_config_overrides.virtual_display_render_env.GALLIUM_NUM_THREADS = int(GALLIUM_NUM_THREADS)
OmegaConf.save(config=cfg, f=colab_cfg)

mod_config = MOD_PATH / "config.yaml"
if mod_config.exists():
    mod_cfg = OmegaConf.load(mod_config)
    mod_cfg.virtual_display = True
    OmegaConf.save(config=mod_cfg, f=mod_config)

print("Wrote:", AGENTARK_ROOT / ".env")
print("Wrote:", colab_cfg)
print("Available tasks:")
tasks = sorted(p.name for p in TASK_STORE_PATH.iterdir() if p.is_dir()) if TASK_STORE_PATH.exists() else []
for name in tasks[:80]:
    print("-", name)
if len(tasks) > 80:
    print(f"... and {len(tasks) - 80} more")

## Start the env server

This is the framework-independent server that RL trainers and debugging clients talk to. It exposes the FastAPI endpoints `/v1/envs/acquire_start`, `/v1/envs/{env_id}/step`, and `/v1/envs/{env_id}/release`.

In [ ]:
# @title Step 3: Start AgentArk env server { display-mode: "form" }

SERVER_PORT = 18080  # @param {type:"integer"}

import os
import subprocess
import time
from pathlib import Path

import requests

AGENTARK_ROOT = Path("/content/AgentArk")
VENV_PY = "/content/agentark_env/bin/python"
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"
SERVER_LOG = Path("/content/agentark_env_server.log")

if "agentark_server_process" in globals() and agentark_server_process.poll() is None:
    print("Stopping previous server process...")
    agentark_server_process.terminate()
    agentark_server_process.wait(timeout=20)

log_f = SERVER_LOG.open("w", encoding="utf-8")
cmd = [VENV_PY, "-m", "agent_ark.ark_env.serving.run_server", "--host", "127.0.0.1", "--port", str(SERVER_PORT)]
agentark_server_process = subprocess.Popen(
    cmd,
    cwd=str(AGENTARK_ROOT),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

for attempt in range(60):
    try:
        r = requests.get(SERVER_URL + "/health", timeout=2)
        if r.ok:
            print("Server is ready:", r.json())
            break
    except Exception:
        time.sleep(1)
else:
    print("Server did not become ready. Last log lines:")
    if SERVER_LOG.exists():
        print("\n".join(SERVER_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
    raise RuntimeError("AgentArk env server failed to start")

print("Server URL:", SERVER_URL)
print("Log file:", SERVER_LOG)

In [ ]:
# @title Step 4: Warm up a small env pool { display-mode: "form" }

WARMUP_NUM_ENVS = 2  # @param {type:"integer"}

import subprocess
from pathlib import Path

AGENTARK_ROOT = Path("/content/AgentArk")
VENV_PY = "/content/agentark_env/bin/python"
CONFIG_PATH = AGENTARK_ROOT / "config" / "ark_env" / "agentark_runtime_config.colab.yaml"
WARMUP_SNAPSHOT = AGENTARK_ROOT / "tmp" / "warmup_snapshot_colab.json"

cmd = [
    VENV_PY,
    "-m",
    "agent_ark.ark_env.serving.warmup_envs",
    "--config",
    str(CONFIG_PATH),
    "--num-envs",
    str(WARMUP_NUM_ENVS),
    "--output",
    str(WARMUP_SNAPSHOT),
]
subprocess.run(cmd, cwd=str(AGENTARK_ROOT), check=True)
print("Warmup snapshot:", WARMUP_SNAPSHOT)

## Framework-free parallel rollout client

The next cell drives the env server with a small async client. It does not import any RL framework. By default it pins two tasks (`Snake,MarbleStop`) and runs multiple envs concurrently, then saves screenshots from each env observation. Leave `TASK_NAMES` empty to demonstrate server-managed task selection: each rollout sends a `uid`, and the server maps that group id to a deterministic `(task_name, group_seed)` pair.

In [ ]:
# @title Step 5: Run framework-free parallel env interaction demo { display-mode: "form" }

TASK_NAMES = "Snake,MarbleStop"  # @param {type:"string"}
ROLLOUTS = 4  # @param {type:"integer"}
CONCURRENCY = 2  # @param {type:"integer"}
MAX_STEPS = 1  # @param {type:"integer"}
ACTION_TEXT = ""  # @param {type:"string"}
MAX_IMAGES_PER_OBSERVATION = 1  # @param {type:"integer"}

import json
import math
import subprocess
from pathlib import Path

AGENTARK_ROOT = Path("/content/AgentArk")
VENV_PY = "/content/agentark_env/bin/python"
CONFIG_PATH = AGENTARK_ROOT / "config" / "ark_env" / "agentark_runtime_config.colab.yaml"
REPORT_PATH = AGENTARK_ROOT / "tmp" / "env_parallel_client_demo_colab.json"
IMAGE_DIR = AGENTARK_ROOT / "tmp" / "env_parallel_client_demo_images_colab"

if REPORT_PATH.exists():
    REPORT_PATH.unlink()

cmd = [
    VENV_PY,
    "-m",
    "agent_ark.tools.env_parallel_client_demo",
    "--config",
    str(CONFIG_PATH),
    "--rollouts",
    str(ROLLOUTS),
    "--concurrency",
    str(CONCURRENCY),
    "--max-steps",
    str(MAX_STEPS),
    "--action",
    ACTION_TEXT,
    "--output",
    str(REPORT_PATH),
    "--image-dir",
    str(IMAGE_DIR),
    "--max-images-per-observation",
    str(MAX_IMAGES_PER_OBSERVATION),
    "--require-valid-config",
]
if TASK_NAMES.strip():
    cmd.extend(["--task-names", TASK_NAMES.strip()])

print("Running:", " ".join(cmd))
completed = subprocess.run(
    cmd,
    cwd=str(AGENTARK_ROOT),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(completed.stdout)

if not REPORT_PATH.exists():
    raise RuntimeError(
        "The env client demo did not write a report. "
        f"Exit code: {completed.returncode}. See output above."
    )

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
print("Report:", REPORT_PATH)
print("ok_count:", report["ok_count"], "error_count:", report["error_count"])
print("server_managed_tasks:", report["requested"]["server_managed_tasks"])
print("task_names:", report["requested"].get("task_names"))
print("image_dir:", report["requested"].get("image_dir"))

for item in report["rollouts"]:
    print(
        f"rollout={item['rollout_index']} ok={item.get('ok')} env={item.get('env_id')} "
        f"task={item.get('requested_task_name')} unity={item.get('unity_id')} "
        f"steps={len(item.get('steps', []))} reward={item.get('total_reward')}"
    )
    if not item.get("ok"):
        print("  error:", item.get("error"))
    if item.get("release_error"):
        print("  release_error:", item.get("release_error"))
    print("  start_obs:", item.get("start_obs"))
    for step in item.get("steps", []):
        print(f"  step {step.get('step_index')} obs:", step.get("obs"))

image_entries = []
for item in report["rollouts"]:
    task = item.get("requested_task_name") or "server-managed"
    rollout = item.get("rollout_index")
    for path in item.get("start_images", []):
        image_entries.append((path, f"rollout {rollout} | {task} | start"))
    for step in item.get("steps", []):
        for path in step.get("images", []):
            image_entries.append((path, f"rollout {rollout} | {task} | step {step.get('step_index')}"))

print(f"Saved {len(image_entries)} screenshots under {IMAGE_DIR}.")

if image_entries:
    from PIL import Image
    import matplotlib.pyplot as plt
    import numpy as np

    show_entries = image_entries[:8]
    cols = min(4, len(show_entries))
    rows = math.ceil(len(show_entries) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (path, title) in zip(axes, show_entries):
        ax.imshow(Image.open(path))
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    for ax in axes[len(show_entries):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No screenshots were saved. Check the start_obs/step obs summaries above for obs['vis'] frame counts and obs['messages'] image counts.")

if report["ok_count"] == 0:
    raise RuntimeError("All demo rollouts failed. See rollout errors above and the env server log from Step 3.")
if completed.returncode != 0:
    print(f"Demo process exited with code {completed.returncode}, but a report was generated and displayed above.")



In [ ]:
# @title Optional: Stop the server { display-mode: "form" }

STOP_SERVER = False  # @param {type:"boolean"}

if STOP_SERVER and "agentark_server_process" in globals() and agentark_server_process.poll() is None:
    agentark_server_process.terminate()
    agentark_server_process.wait(timeout=20)
    print("Server stopped.")
else:
    print("Server left running for further experiments.")

## Part 2: RL framework integration runbooks

Full training runs outside Colab in a framework-specific trainer environment. The Unity runtime and AgentArk Env Server remain separate and communicate with the trainer over HTTP.

Choose one maintained path:

- [ms-swift GRPO runbook](../../integrations/ms_swift/README.md): the repository-local Swift Gym adapter, protocol v2 warmup, smoke, and training launcher.
- [VERL GRPO integration guide](../../integrations/verl/README.md): checkout compatibility check, exact-config protocol v1 warmup, dataset generation, and the guarded external launcher.
- [RL architecture and semantics](../rl-training.md): process boundaries, within-batch async interaction, GRPO task/seed grouping, and v1/v2 recovery differences.

Do not reuse Part 1's small, generic Colab warmup config as a framework training pool. v1 and v2 namespaces are isolated, and runtime reuse also depends on the semantic environment-config fingerprint. Follow the selected runbook so the Server is warmed with the exact config and capacity that its adapter sends.

For a small task-specific correctness experiment such as Snake 8x8, first complete the selected runbook's general smoke. Then use a separate task store or runtime Mods directory, pin `Snake` and a seed in that framework's dataset, reduce `task_params.gridWidth/gridHeight` to 8, and rerun the same preflight and launcher. This keeps the original multi-task store recoverable and makes the task-specific change explicit.


## Training Figures

Large training screenshots are intentionally not embedded in this public notebook to keep the file lightweight. Use the logging and checkpoint settings in the selected framework runbook to generate fresh console, TensorBoard, or experiment-tracking plots for your own run.
